# M02 · KNN (k vecinos más cercanos)

Clasifica cada producto según la clase de sus k vecinos más parecidos. Como depende de distancias,
el escalado es imprescindible: usamos el mismo preprocesado que la M01 (`src/modeling.py`).

**Reglas del equipo (EDA, paso 14.4):** solo `train`; CV con `StratifiedGroupKFold` por `grupo_producto`;
F1 macro + recall por clase; gap train/CV < 0,05.

**Experimentos:**
1. Baseline: k=5, pesos uniformes, distancia euclídea.
2. Búsqueda de `n_neighbors`, `weights` y `p` con `GridSearchCV`.
3. Variantes: sin `energy-kcal_100g` y `saturated-fat_100g`.
4. Análisis de errores y comparación con M01 (4a: F1 macro CV 0,9508).

In [1]:
import sys
sys.path.insert(0, "..")

import json
import platform
import time

import joblib
import pandas as pd
import sklearn
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

from src.modeling import (TARGET, GROUP, LABELS, load_train, get_feature_columns,
                          build_preprocessor, evaluate_cv, MODELS_DIR, RESULTS_DIR)

In [2]:
df = load_train()
print(df.shape)
print(df[TARGET].value_counts(normalize=True).sort_index().round(3))

DROP_REDUNDANT = ["energy-kcal_100g", "saturated-fat_100g"]
y, groups = df[TARGET], df[GROUP]

(38495, 35)
nova_group
1    0.129
2    0.112
3    0.204
4    0.555
Name: proportion, dtype: float64


## Función auxiliar

Un solo sitio para construir y evaluar cada variante, como en la M01.
Guardamos también la configuración y el tiempo, porque KNN es lento al predecir.

In [3]:
results = {}

def run(name, n_neighbors=5, weights="uniform", p=2, drop=()):
    cols = get_feature_columns(df, drop=drop)
    model = Pipeline([
        ("prep", build_preprocessor(cols, log_skewed=True)),   # log1p + escalado: imprescindible en KNN
        ("clf", KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights, p=p)),
    ])
    start = time.time()
    res = evaluate_cv(model, df[cols["all"]], y, groups)
    res["seconds"] = round(time.time() - start, 1)
    res["config"] = {"n_neighbors": n_neighbors, "weights": weights, "p": p, "dropped": list(drop)}
    results[name] = res
    print(f"{name}: F1 macro CV {res['f1_macro_cv']} (±{res['f1_macro_cv_std']}), "
          f"gap {res['gap']}, recall {res['recall']}, {res['seconds']} s")
    return res

## 1. Baseline: k=5, pesos uniformes, distancia euclídea

Valores por defecto de scikit-learn. Referencias: Dummy ≈ 0,178 · M01 (4a) 0,9508.

In [4]:
run("1_baseline_k5")

1_baseline_k5: F1 macro CV 0.921 (±0.0046), gap 0.0233, recall {'nova_1': 0.9416, 'nova_2': 0.9814, 'nova_3': 0.8795, 'nova_4': 0.924}, 13.0 s


{'f1_macro_train': 0.9443,
 'f1_macro_cv': 0.921,
 'f1_macro_cv_std': 0.0046,
 'gap': 0.0233,
 'recall': {'nova_1': 0.9416,
  'nova_2': 0.9814,
  'nova_3': 0.8795,
  'nova_4': 0.924},
 'confusion_matrix': [[4663, 36, 180, 73],
  [27, 4228, 24, 29],
  [306, 33, 6917, 609],
  [261, 38, 1326, 19745]],
 'seconds': 13.0,
 'config': {'n_neighbors': 5, 'weights': 'uniform', 'p': 2, 'dropped': []}}

## 2. Búsqueda de `n_neighbors`, `weights` y `p`

- `n_neighbors`: con k pequeño la votación es ruidosa; con k grande, las clases minoritarias quedan tapadas por NOVA 4.
- `weights="distance"`: los vecinos más cercanos pesan más. Es la alternativa a `class_weight`, que KNN no tiene.
- `p`: 1 = Manhattan, 2 = euclídea. Manhattan suele ir mejor con muchas dimensiones.

Mismos folds que `evaluate_cv` (`StratifiedGroupKFold`, `random_state=42`).

In [5]:
cols = get_feature_columns(df)
pipe = Pipeline([
    ("prep", build_preprocessor(cols, log_skewed=True)),
    ("clf", KNeighborsClassifier()),
])

param_grid = {
    "clf__n_neighbors": [5, 11, 21, 31, 51],
    "clf__weights": ["uniform", "distance"],
    "clf__p": [1, 2],
}

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)  # los mismos folds que evaluate_cv
grid = GridSearchCV(pipe, param_grid, cv=cv, scoring="f1_macro",
                    return_train_score=True, n_jobs=-1)

start = time.time()
grid.fit(df[cols["all"]], y, groups=groups)
print(f"{time.time() - start:.0f} s · mejor: {grid.best_params_} → {grid.best_score_:.4f}")

328 s · mejor: {'clf__n_neighbors': 5, 'clf__p': 1, 'clf__weights': 'distance'} → 0.9333


In [6]:
grid_table = (pd.DataFrame(grid.cv_results_)
              .assign(gap=lambda d: d["mean_train_score"] - d["mean_test_score"])
              [["param_clf__n_neighbors", "param_clf__weights", "param_clf__p",
                "mean_test_score", "std_test_score", "mean_train_score", "gap"]]
              .sort_values("mean_test_score", ascending=False)
              .round(4))
grid_table

,param_clf__n_neighbors,param_clf__weights,param_clf__p,mean_test_score,std_test_score,mean_train_score,gap
1,5,distance,1,0.9333,0.0022,0.9997,0.0664
5,11,distance,1,0.9323,0.0040,0.9997,0.0674
9,21,distance,1,0.9288,0.0029,0.9997,0.0708
0,5,uniform,1,0.9280,0.0031,0.9500,0.0219
3,5,distance,2,0.9272,0.0037,0.9997,0.0725
13,31,distance,1,0.9258,0.0034,0.9997,0.0739
7,11,distance,2,0.9241,0.0044,0.9997,0.0756
4,11,uniform,1,0.9227,0.0045,0.9353,0.0126
2,5,uniform,2,0.9210,0.0046,0.9443,0.0233
17,51,distance,1,0.9202,0.0023,0.9997,0.0795


### Ampliación hacia k pequeños

El mejor `k` quedó en el borde de la rejilla (5). Probamos valores más pequeños
con la combinación ganadora (`p=1`, `weights="distance"`).
k=1 es el caso extremo: la clase del vecino más parecido.

**Criterio:** elegimos el k más grande cuyo F1 macro CV esté a menos de una std del mejor
(fronteras más suaves y más estables ante productos nuevos).

In [7]:
grid_small = GridSearchCV(
    pipe,
    {"clf__n_neighbors": [1, 3, 5, 7, 9], "clf__p": [1], "clf__weights": ["distance"]},
    cv=cv, scoring="f1_macro", n_jobs=-1,
)
start = time.time()
grid_small.fit(df[cols["all"]], y, groups=groups)
print(f"{time.time() - start:.0f} s · mejor: {grid_small.best_params_} → {grid_small.best_score_:.4f}")

(pd.DataFrame(grid_small.cv_results_)
 [["param_clf__n_neighbors", "mean_test_score", "std_test_score"]]
 .round(4))

30 s · mejor: {'clf__n_neighbors': 7, 'clf__p': 1, 'clf__weights': 'distance'} → 0.9341


,param_clf__n_neighbors,mean_test_score,std_test_score
0,1,0.9278,0.0030
1,3,0.9331,0.0034
2,5,0.9333,0.0022
3,7,0.9341,0.0031
4,9,0.9331,0.0043


**Elección:** k=11, Manhattan (`p=1`), `weights="distance"`.
El mejor F1 CV es k=7 (0,9341 ± 0,0031); k=9 y k=11 quedan dentro de una std. Aplicando el criterio, k=11.
De k=3 a k=11 el F1 es prácticamente igual (meseta); k=1 baja porque un solo vecino es muy ruidoso.

## 3. Modelo elegido y variante sin variables redundantes

In [8]:
BEST = {"n_neighbors": 11, "weights": "distance", "p": 1}

_ = run("2_k11_manhattan_distance", **BEST)
_ = run("3_k11_sin_redundantes", **BEST, drop=DROP_REDUNDANT)

2_k11_manhattan_distance: F1 macro CV 0.9323 (±0.004), gap 0.0674, recall {'nova_1': 0.9515, 'nova_2': 0.9861, 'nova_3': 0.9041, 'nova_4': 0.9299}, 49.1 s
3_k11_sin_redundantes: F1 macro CV 0.9351 (±0.0036), gap 0.0646, recall {'nova_1': 0.9536, 'nova_2': 0.9877, 'nova_3': 0.9102, 'nova_4': 0.9316}, 48.6 s


**Elegimos la variante 3** (sin `energy-kcal_100g` ni `saturated-fat_100g`): +0,003 de F1 (dentro de la std),
pero mejoran las cuatro clases y usa las mismas variables que la M01 (4a).

## 4. Análisis de errores frente a la M01

In [9]:
FINAL = "3_k11_sin_redundantes"
m01 = json.loads((RESULTS_DIR / "M01-metrics.json").read_text())

names = [f"NOVA {k}" for k in LABELS]
cm_knn = pd.DataFrame(results[FINAL]["confusion_matrix"], index=names, columns=names)
cm_m01 = pd.DataFrame(m01["cv"]["confusion_matrix"], index=names, columns=names)

print("KNN (filas = real, columnas = predicha)")
display(cm_knn)
print("Diferencia KNN − M01 (positivo = KNN se equivoca más)")
display(cm_knn - cm_m01)

KNN (filas = real, columnas = predicha)


,NOVA 1,NOVA 2,NOVA 3,NOVA 4
NOVA 1,4722,29,154,47
NOVA 2,23,4255,20,10
NOVA 3,238,24,7159,444
NOVA 4,214,27,1221,19908


Diferencia KNN − M01 (positivo = KNN se equivoca más)


,NOVA 1,NOVA 2,NOVA 3,NOVA 4
NOVA 1,-127,1,114,12
NOVA 2,2,2,0,-4
NOVA 3,5,2,-117,110
NOVA 4,19,17,347,-383


**Lectura de la diferencia con M01:** NOVA 2 y los errores 3 → 1 son prácticamente iguales (mismos productos
difíciles en los dos modelos). KNN pierde sobre todo hacia NOVA 3: +347 NOVA 4 → 3 y +114 NOVA 1 → 3.
NOVA 3 y 4 se parecen en nutrientes y se separan por los marcadores (aromas, emulsionantes, aditivos);
KNN da el mismo peso a todas las columnas, así que esos marcadores se diluyen en la distancia.

## 5. Resumen de variantes

In [10]:
table = pd.DataFrame({
    name: {"F1 macro train": r["f1_macro_train"], "F1 macro CV": r["f1_macro_cv"],
           "std": r["f1_macro_cv_std"], "gap": r["gap"], "segundos": r["seconds"],
           **{f"recall {k}": v for k, v in r["recall"].items()}}
    for name, r in results.items()
}).T
table.sort_values("F1 macro CV", ascending=False)

,F1 macro train,F1 macro CV,std,gap,segundos,recall nova_1,recall nova_2,recall nova_3,recall nova_4
3_k11_sin_redundantes,0.9997,0.9351,0.0036,0.0646,48.6,0.9536,0.9877,0.9102,0.9316
2_k11_manhattan_distance,0.9997,0.9323,0.0040,0.0674,49.1,0.9515,0.9861,0.9041,0.9299
1_baseline_k5,0.9443,0.9210,0.0046,0.0233,13.0,0.9416,0.9814,0.8795,0.9240


## Guardar el modelo y las métricas

Entrenamos el modelo elegido con **todo el train**. Las métricas guardadas son las de la validación cruzada.

- Modelo: `models/m02_knn.joblib` (ignorado por git; se regenera con este notebook).
- Métricas: `results/M02-metrics.json`.

**Ojo con el gap:** con `weights="distance"` el F1 de train es ≈ 1, porque cada producto de train es su propio
vecino a distancia 0. El gap (0,065) no mide sobreajuste en KNN; con `weights="uniform"` y la misma `k`
el gap baja a ≈ 0,01 con un F1 CV casi igual (tabla del paso 2).

In [12]:
cols_final = get_feature_columns(df, drop=DROP_REDUNDANT)
final = Pipeline([
    ("prep", build_preprocessor(cols_final, log_skewed=True)),
    ("clf", KNeighborsClassifier(**BEST)),
])
final.fit(df[cols_final["all"]], y)

MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
joblib.dump(final, MODELS_DIR / "m02_knn.joblib")

metrics = {
    "model": "KNeighborsClassifier",
    "variant": FINAL,
    "config": {**BEST, "log_skewed": True, "dropped": DROP_REDUNDANT},
    "features": cols_final["all"],
    "cv": results[FINAL],
    "dummy_cv": m01["dummy_cv"],          # el suelo es el mismo para todos: lo tomamos de M01
    "grid_search": {                      # para M08: qué se probó y por qué k=11
        "param_grid": {k.replace("clf__", ""): v for k, v in param_grid.items()},
        "best_cv": {k.replace("clf__", ""): v for k, v in grid.best_params_.items()},
        "ampliacion_k": [1, 3, 5, 7, 9],
        "best_cv_ampliacion": {k.replace("clf__", ""): v for k, v in grid_small.best_params_.items()},
        "best_f1_cv_ampliacion": round(float(grid_small.best_score_), 4),
        "criterio_k": "k más grande a menos de 1 std del mejor F1 CV (k=7: 0,9341 ± 0,0031 → k=11)",
    },
    "notes": "Con weights='distance' el F1 de train es ~1 (cada punto es su propio vecino): el gap no mide sobreajuste.",
    "n_train": len(df),
    "env": {"python": platform.python_version(), "scikit-learn": sklearn.__version__},
}
with open(RESULTS_DIR / "M02-metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print("Guardado:", MODELS_DIR / "m02_knn.joblib")

Guardado: /app/models/m02_knn.joblib


## Conclusiones

- **Suelo:** el Dummy (siempre NOVA 4) tiene F1 macro 0,1785. KNN lo supera de largo.
- **KNN llega a un F1 macro de 0,9351 en CV** (k=11, Manhattan, `weights="distance"`, sin `energy-kcal_100g`
  ni `saturated-fat_100g`). El baseline por defecto (k=5, euclídea, uniforme) daba 0,9210.
- **Lo que más mejora:** la distancia Manhattan (`p=1`) gana a la euclídea en todas las combinaciones, y
  `weights="distance"` gana a `"uniform"`. Con muchas columnas, Manhattan reparte mejor las diferencias.
- **El valor de `k` apenas importa entre 3 y 11** (F1 entre 0,932 y 0,934, dentro de la std). k=1 baja (0,928)
  y a partir de k=21 también. Elegimos k=11 con el criterio de "k más grande a menos de una std del mejor".
- **El gap (0,065) no indica sobreajuste:** con `weights="distance"` cada producto de train es su propio vecino
  a distancia 0, así que el F1 de train es ≈ 1. Con `weights="uniform"` el gap baja a ≈ 0,01 con un F1 CV casi igual.
- **Queda 0,016 por debajo de la M01 (0,9508)** y no la supera en ninguna clase. KNN da el mismo peso a todas
  las columnas; la regresión logística aprende cuáles importan.
- **Dónde falla:** casi todo el error de más va hacia NOVA 3 (+347 NOVA 4 → 3 y +114 NOVA 1 → 3 respecto a M01).
  NOVA 3 y 4 tienen nutrientes parecidos y se separan por los marcadores (aromas, emulsionantes, aditivos),
  que en la distancia se diluyen entre ~60 columnas.
- **Lo que comparte con M01:** NOVA 2 se reconoce igual de bien (recall 0,988) y los errores NOVA 3 → 1 son
  casi los mismos (238 frente a 233): productos difíciles para cualquier modelo, quizá mal etiquetados en OFF.
- **Coste:** el modelo guarda todo el train (22 MB frente a 10 KB de la M01) y predecir es lento
  (≈ 50 s por validación cruzada). Para la app no es un problema, pero cuenta en la comparación de M08.
- El test no se ha usado.